<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Акцентологическая-разметка" data-toc-modified-id="Акцентологическая-разметка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Акцентологическая разметка</a></span></li><li><span><a href="#Определение-метра" data-toc-modified-id="Определение-метра-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Определение метра</a></span><ul class="toc-item"><li><span><a href="#Обработка-с-выводом-статистики" data-toc-modified-id="Обработка-с-выводом-статистики-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Обработка с выводом статистики</a></span></li></ul></li><li><span><a href="#Распределение-размеченных-строк-по-файлам" data-toc-modified-id="Распределение-размеченных-строк-по-файлам-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Распределение размеченных строк по файлам</a></span><ul class="toc-item"><li><span><a href="#Чтение-файла-и-распределение-по-метрам" data-toc-modified-id="Чтение-файла-и-распределение-по-метрам-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Чтение файла и распределение по метрам</a></span></li></ul></li></ul></div>

## Акцентологическая разметка

In [1]:
from ru_accent_poet import accent_line

In [3]:
from tqdm import tqdm

In [2]:
accent_line('Это инструмент для разметки ударений') # тест

"Э'то инструме'нт для разме'тки ударе'ний"

In [17]:
fw = open('verses_accented.txt', 'w')
with open('verses.txt', 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Обработка стихов", unit="строк"):
        try:
            la = accent_line(line)
            fw.write(la + '\n')
        except IndexError as e:
            # Если слово слишком длинное, оставляем строку без изменений
            print(f"\nПредупреждение: пропущена строка из-за слишком длинного слова: {line[:50]}...")
            fw.write(line)  # Записываем исходную строку
        except Exception as e:
            continue

fw.close()

Обработка стихов: 14207строк [29:58,  7.76строк/s]


Предупреждение: пропущена строка из-за слишком длинного слова: Которым я в Жан д'Арке восхищаюсь,
...


Обработка стихов: 114628строк [4:26:29,  7.17строк/s]


## Определение метра

In [6]:
import re

In [18]:
def analyze_line(line):
    """Анализирует строку и возвращает подробную информацию"""
    vowels = 'аеёиоуыэюя'
    
    # Подсчет гласных
    clean_line = line.replace("'", "")
    total_vowels = sum(1 for char in clean_line.lower() if char in vowels)
    
    # Поиск ударений
    stressed = []
    syllable = 1
    i = 0
    chars = list(line)
    
    while i < len(chars):
        char = chars[i].lower()
        if char in vowels:
            if char == 'ё' or (i + 1 < len(chars) and chars[i + 1] == "'"):
                stressed.append(syllable)
            syllable += 1
        i += 1
    
    # Определение метра
    if not stressed:
        return "Не определено", 0, stressed, total_vowels
    
    # Ямб
    if all(p % 2 == 0 for p in stressed):
        meter = "Я"
        feet = total_vowels // 2
    # Хорей
    elif all(p % 2 == 1 for p in stressed):
        meter = "Х"
        feet = (total_vowels + 1) // 2
    # Дактиль
    elif all((p - 1) % 3 == 0 for p in stressed):
        meter = "Д"
        feet = (total_vowels + 2) // 3
    # Амфибрахий
    elif all((p - 2) % 3 == 0 for p in stressed):
        meter = "Аф"
        feet = (total_vowels + 1) // 3
    # Анапест
    elif all((p - 3) % 3 == 0 for p in stressed):
        meter = "Ан"
        feet = total_vowels // 3
    else:
        meter = "Не определено"
        feet = 0
    
    return meter, max(feet, 1) if meter != "Не определено" else 0, stressed, total_vowels



### Обработка с выводом статистики

In [19]:
input_file = 'verses_accented.txt'
output_file = 'verses_with_meter.txt'

In [20]:
stats = {"Я": 0, "Х": 0, "Д": 0, "Аф": 0, "Ан": 0, "Не определено": 0}

with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out:
    
    for line_num, line in enumerate(f_in, 1):
        line = line.rstrip('\n')
        
        if line.strip():
            meter, feet, stressed, total_vowels = analyze_line(line)
            stats[meter] += 1
            
            output_line = f"<{meter}{feet}>{line}\n"
            
            # Для отладки (можно закомментировать)
            #if stressed:
                #print(f"Строка {line_num}: слогов={total_vowels}, ударения={stressed}, метр={meter}{feet}")
        else:
            output_line = '\n'
        
        f_out.write(output_line)
        
        if line_num % 10000 == 0:
            print(f"Обработано строк: {line_num}")

# Вывод статистики
print("\n" + "="*50)
print("СТАТИСТИКА ПО СТРОКАМ:")
print("="*50)
for meter, count in stats.items():
    if count > 0:
        print(f"{meter}: {count} строк")
print("="*50)
print(f"\nРезультат сохранен в файл '{output_file}'")

Обработано строк: 10000
Обработано строк: 20000
Обработано строк: 30000
Обработано строк: 40000
Обработано строк: 50000
Обработано строк: 60000
Обработано строк: 70000
Обработано строк: 80000
Обработано строк: 90000
Обработано строк: 100000
Обработано строк: 110000

СТАТИСТИКА ПО СТРОКАМ:
Я: 52670 строк
Х: 13022 строк
Д: 4719 строк
Аф: 5295 строк
Ан: 3251 строк
Не определено: 15694 строк

Результат сохранен в файл 'verses_with_meter.txt'


## Распределение размеченных строк по файлам

In [13]:
import os
from collections import defaultdict

In [21]:
def parse_meter_tag(line):
    """Извлекает метр и число стоп из тега в начале строки"""
    # Ищем тег в формате <МетрЧисло>
    match = re.match(r'<([А-Яа-я]+)(\d+)>(.*)', line)
    if match:
        meter = match.group(1)  # Название метра
        feet = match.group(2)   # Число стоп
        content = match.group(3)  # Остальная часть строки
        return meter, feet, content
    return None, None, line

def remove_apostrophes(text):
    """Удаляет все апострофы из текста"""
    return text.replace("'", "")

def count_words(text):
    """Подсчитывает количество слов в строке"""
    # Разбиваем по пробелам и удаляем пустые строки
    words = [w for w in text.split() if w.strip()]
    return len(words)


### Чтение файла и распределение по метрам

In [22]:
input_file = 'verses_with_meter.txt'
output_dir = './metersRSD'  # Директория для строк с метром

In [23]:
# Словарь для хранения строк по метрам и стопам
# Структура: {('Я', '4'): ['строка1', 'строка2'], ...}
lines_by_meter = defaultdict(list)

# Словарь для статистики слов
word_stats = defaultdict(int)

print("Чтение файла...")

with open(input_file, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.rstrip('\n')
        
        if not line.strip():  # Пропускаем пустые строки
            continue
        
        # Извлекаем метр и число стоп
        meter, feet, content = parse_meter_tag(line)
        
        if meter and feet:
            # Удаляем апострофы
            clean_content = remove_apostrophes(content)
            
            # Сохраняем в словарь
            key = (meter, feet)
            lines_by_meter[key].append(clean_content)
            
            # Подсчитываем слова
            word_count = count_words(clean_content)
            word_stats[key] += word_count
        else:
            continue
            #print(f"Предупреждение: строка {line_num} не содержит корректного тега: {line[:50]}...")

print(f"Найдено {len(lines_by_meter)} различных комбинаций метра и стоп")

# Создаем файлы для каждой комбинации
print("\nСоздание файлов...")

for (meter, feet), lines in lines_by_meter.items():
    filename = f"{meter}{feet}.txt"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f_out:
        for line in lines:
            f_out.write(line + '\n')
    
    print(f"Создан файл {filename}: {len(lines)} строк, {word_stats[(meter, feet)]} слов")

# Вывод финальной статистики
print("\n" + "="*60)
print("ФИНАЛЬНАЯ СТАТИСТИКА ПО ФАЙЛАМ")
print("="*60)
print(f"{'Файл':<20} {'Строки':<10} {'Слова':<10}")
print("-"*60)

for (meter, feet), lines in sorted(lines_by_meter.items()):
    filename = f"{meter}{feet}.txt"
    word_count = word_stats[(meter, feet)]
    print(f"{filename:<20} {len(lines):<10} {word_count:<10}")

print("="*60)

# Дополнительная статистика по метрам
print("\n" + "="*60)
print("СТАТИСТИКА ПО МЕТРАМ")
print("="*60)

meter_stats = defaultdict(lambda: {'files': 0, 'lines': 0, 'words': 0})

for (meter, feet), lines in lines_by_meter.items():
    meter_stats[meter]['files'] += 1
    meter_stats[meter]['lines'] += len(lines)
    meter_stats[meter]['words'] += word_stats[(meter, feet)]

print(f"{'Метр':<15} {'Файлов':<8} {'Строк':<10} {'Слов':<10}")
print("-"*60)

for meter, stats in sorted(meter_stats.items()):
    print(f"{meter:<15} {stats['files']:<8} {stats['lines']:<10} {stats['words']:<10}")

print("="*60)
print("\nГотово! Файлы созданы в текущей директории.")

Чтение файла...
Найдено 36 различных комбинаций метра и стоп

Создание файлов...
Создан файл Я4.txt: 28560 строк, 121351 слов
Создан файл Д3.txt: 2186 строк, 8329 слов
Создан файл Я2.txt: 1058 строк, 2688 слов
Создан файл Х4.txt: 8801 строк, 34525 слов
Создан файл Я3.txt: 7912 строк, 26816 слов
Создан файл Ан2.txt: 591 строк, 2151 слов
Создан файл Я6.txt: 6804 строк, 43168 слов
Создан файл Х2.txt: 538 строк, 1249 слов
Создан файл Аф4.txt: 1981 строк, 11440 слов
Создан файл Я5.txt: 7736 строк, 43132 слов
Создан файл Я1.txt: 577 строк, 1140 слов
Создан файл Аф2.txt: 833 строк, 2436 слов
Создан файл Аф3.txt: 2405 строк, 10270 слов
Создан файл Ан3.txt: 2467 строк, 11317 слов
Создан файл Д2.txt: 376 строк, 921 слов
Создан файл Х3.txt: 1173 строк, 3410 слов
Создан файл Х1.txt: 216 строк, 459 слов
Создан файл Д4.txt: 1835 строк, 9561 слов
Создан файл Аф5.txt: 60 строк, 433 слов
Создан файл Д5.txt: 103 строк, 695 слов
Создан файл Д6.txt: 213 строк, 1750 слов
Создан файл Х5.txt: 1985 строк, 887